# Generate Training Crops for CNN

This notebook extracts image crops from raw microscopy .lif files for CNN training.

## Workflow
```
1. Generate_Training_Crops.ipynb  ← YOU ARE HERE
   └── Creates: training/training_crops_*/

2. Human_Labeling_Tool.ipynb
   └── Labels crops → training/human_selection_*.npy

3. ML_Tranining_Validation.ipynb
   └── Trains models → spot_detection_cnn.pth
```

## Output
- `training/training_crops_real_data/` - Crops from real microscopy
- `training/training_crops_simulated_data/` - Synthetic spots

In [ ]:
"""
MicroLive Notebook
==================
This notebook requires MicroLive to be installed:
    pip install microlive

For development mode:
    pip install -e /path/to/microlive
"""
# MicroLive imports
from microlive import microscopy as mi
from microlive.utils.device import check_gpu_status

# Verify GPU support
check_gpu_status()

# Standard scientific imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


## Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Update these paths for your data
# =============================================================================

# Path to raw microscopy data
DATA_FOLDER = Path('/Users/nzlab-la/Library/CloudStorage/OneDrive-TheUniversityofColoradoDenver/General - Zhao (NZ) Lab/Microscope/Luis Aguilera/Live cell imaging_Folding & Nascent chains')

# Leica .lif file to process
LIF_FILENAME = '20240806 pNZ212 and pRS001_JF646_NoDelay_ch0 folding_ch1 nascent chains.lif'

# Which image in the .lif file (0-indexed)
SELECTED_IMAGE_INDEX = 4

# Crop settings
CROP_SIZE = 11
COLOR_CHANNEL = 0

# Output folders (relative to this notebook)
OUTPUT_REAL = Path('training/training_crops_real_data')
OUTPUT_SIMULATED = Path('training/training_crops_simulated_data')
OUTPUT_HUMAN = Path('training/training_crops_human_selection')

print("Configuration set!")
print(f"Data folder exists: {DATA_FOLDER.exists()}")

## 1. Load Raw Microscopy Data

In [ ]:
# Load the .lif file
data_path = DATA_FOLDER / LIF_FILENAME

print(f'Loading: {data_path}')

# ReadLif returns 11 values
(list_images, list_names, pixel_xy_um, voxel_z_um, 
 channel_names, number_color_channels, list_time_intervals, bit_depth,
 list_laser_lines, list_intensities, list_wave_ranges) = mi.ReadLif(
    data_path,
    show_metadata=True,
    save_tif=False,
    save_png=False,
    format='TZYXC'
).read()

# Select the image
real_image = list_images[SELECTED_IMAGE_INDEX]
print(f'\nSelected: {list_names[SELECTED_IMAGE_INDEX]}')
print(f'Shape: {real_image.shape}')

## 2. Preprocess Image

In [ ]:
# Normalize image to remove extreme values
number_time_points = real_image.shape[0]
filtered_image = np.zeros_like(real_image, dtype=np.float32)

for t in range(number_time_points):
    filtered_image[t] = mi.RemoveExtrema(real_image[t], min_percentile=0.5, max_percentile=99).remove_outliers()

print(f"Preprocessed {number_time_points} time points")
print(f"Value range: {filtered_image.min():.1f} - {filtered_image.max():.1f}")

## 3. Extract Crops (TODO)

This section needs tracking data to extract crops around detected spots.

In [ ]:
# TODO: Extract crops and save to training/ folder
# See dev/MachineLearning_spot_detection.ipynb for reference implementation

print("Crops are already generated in training/ folder")
print(f"training_crops_real_data: {len(list(OUTPUT_REAL.glob('*.png')))} crops")
print(f"training_crops_simulated_data: {len(list(OUTPUT_SIMULATED.glob('*.png')))} crops")
print(f"training_crops_human_selection: {len(list(OUTPUT_HUMAN.glob('*.png')))} crops")

## Next Steps

1. Run `Human_Labeling_Tool.ipynb` to label crops
2. Run `ML_Tranining_Validation.ipynb` to train models